In [ ]:
from rich.console import Console
from rich_theme_manager import Theme, ThemeManager
import pathlib

theme_dir = pathlib.Path("themes")
theme_manager = ThemeManager(theme_dir=theme_dir)
dark = theme_manager.get("dark")

# Create a console with the dark theme
console = Console(theme=dark)

In [ ]:
print(theme_dir.resolve())
print(theme_dir.exists())

In [ ]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
from datasets import load_dataset

dataset = load_dataset("jamescalam/ai-arxiv2", split="train")
console.print(dataset)

### InMemoryVectorStore
In-memory vector store implementation.

Uses a dictionary, and computes cosine similarity for search using numpy.

In [ ]:
import os   
from dotenv import load_dotenv
load_dotenv()
console.print(os.environ['GROQ_API_KEY'])


In [ ]:
from langchain.chat_models import init_chat_model

llm=init_chat_model("groq:openai/gpt-oss-120b")
console.print(llm)


In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.vectorstores import InMemoryVectorStore

vector_store=InMemoryVectorStore(embedding=HuggingFaceEmbeddings())

In [ ]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
console.print(documents)

In [ ]:
console.print(vector_store.add_documents(documents=documents))

In [ ]:
console.print(vector_store.similarity_search("hows the weather forecast"))

In [ ]:
console.print(vector_store.similarity_search("hows the weather forecast",k=2))

In [ ]:
### vectorstore to retriever

retriever=vector_store.as_retriever(search_kwargs={"k":2})

console.print(retriever)

In [ ]:
## Invoke
console.print(
retriever.invoke("hows the weather forecast"))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
## text splitting
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(documents)
console.print(chunks[0])
console.print(chunks[1])

In [ ]:
console.print(f"Created {len(chunks)} chunks from {len(documents)} documents")
console.print("\nExample chunk:")
console.print(f"Content: {chunks[0].page_content}")
console.print(f"Metadata: {chunks[0].metadata}")

In [ ]:
# Initialize OpenAI embeddings with the latest model

embeddings=HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
texts = [doc.page_content for doc in documents]

sentence_embeddings = embeddings.embed_documents(texts)

console.print(len(sentence_embeddings))
console.print(sentence_embeddings[0])
console.print(sentence_embeddings[1])

In [ ]:
### Compare Embedding using cosine similarity
import numpy as np
def compare_embeddings(text1:str,text2:str):
    """Compare semantic simialrity of 2 texts usign embeddings"""

    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    ## Calculate the simialrity score

    similarity=np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [ ]:
# Test semantic similarity
console.print("\nSemantic Similarity Examples:")
console.print(f"'soccer players' vs 'iphone': {compare_embeddings('soccer players', 'iphone'):.3f}")

In [ ]:
console.print(f"LangGraph' vs 'Langchain': {compare_embeddings('LangGraph', 'Lanchain'):.3f}")

In [ ]:
### Similarity Search with score
query="hows the weather forecast"
results_with_scores=vector_store.similarity_search_with_score(query,k=3)

console.print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    console.print(f"\nScore: {score:.3f}")
    console.print(f"Source: {doc.metadata['source']}")
    console.print(f"Content preview: {doc.page_content[:100]}...")

In [ ]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")
    print(f"   Score: {score:.3f}")

In [ ]:
# 1. Simple RAG Chain with LCEL
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [ ]:
## Basic retriever
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
retriever

In [ ]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
simple_rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough() }
    | simple_prompt
    | llm
    |StrOutputParser()

)

In [ ]:
console.print(simple_rag_chain)

In [ ]:
### Conversational RAg Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [ ]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [ ]:
conversational_rag

In [ ]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

In [ ]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [ ]:
test_rag_chains("What is the difference between scrambled eggs and iphone")

In [ ]:
# Test with multiple questions
test_questions = [
    "What is the difference between scrambled eggs and iphone?",
    "Explain LangGraph and Langchain in simple terms",
    "How does weather forecast work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)

In [ ]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is Lanchain?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")

In [ ]:
# Update history
from langchain_core.messages import HumanMessage, AIMessage
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [ ]:
# Follow-up question
q2 = "How is it different Huggingface?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")